# 🛠️ Argo on Colab — Qwen3-14B-abliterated (رایگان، GPU)

این نوت‌بوک Argo رو روی T4 GPU بالا میاره. **بدون کامپایل** — فقط `pip install` با wheel آماده.

## ⚠️ نکته مهم درباره حافظه

Qwen3-14B (Q4_K_M) با context 8K حدود **11GB VRAM** می‌گیره. T4 فقط 16GB داره. **پس context رو 8K گذاشتیم** (نه 12K) تا جا برای overhead Colab (~2GB) و KV cache (~3GB) باشه.

## راه‌اندازی

1. **Runtime → Change runtime type → T4 GPU**
2. سلول‌ها رو به ترتیب **Shift+Enter** اجرا کن
3. آخرین سلول یه URL عمومی میده — روش بزن و چت کن

## چقدر طول می‌کشه؟

- بار اول: ~3 دقیقه (نصب wheel + دانلود 9GB مدل)
- بار دوم (با Drive cache): ~30 ثانیه

## 1) GPU check

In [ ]:
!nvidia-smi | head -10
!echo '---'
!free -h | head -3
!df -h / | tail -1

## 2) نصب بدون کامپایل (~30 ثانیه)

از **wheel آماده** `llama-cpp-python` با CUDA 12.4 می‌گیریم. هیچ کامپایلی نیست.

In [ ]:
import os, subprocess, sys, time

t0 = time.time()
print('📦 Installing llama-cpp-python (prebuilt wheel with CUDA 12.4)...')
try:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'llama-cpp-python',
        '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu124',
        '--prefer-binary',
    ], check=True)
    print(f'✅ llama-cpp-python installed in {time.time()-t0:.0f}s (prebuilt wheel)')
except subprocess.CalledProcessError as e:
    print(f'⚠️  Wheel install failed: {e}')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'llama-cpp-python==0.3.4',
        '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu124',
    ], check=True)

print('\n📦 Installing other deps...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi', 'uvicorn[standard]', 'huggingface_hub', 'pydantic',
], check=True)

import llama_cpp
print(f'\n✅ llama-cpp-python v{llama_cpp.__version__} ready')
print('✅ pip deps OK. (Skipping GPU test — risky on T4, can crash kernel if OOM)')

## 3) دانلود Qwen3-14B-abliterated (Q4_K_M, ~9GB)

مدل رو روی Google Drive کش می‌کنیم تا دفعه بعد ~30 ثانیه طول بکشه.

از **`bartowski/huihui-ai_Qwen3-14B-abliterated-GGUF`** استفاده می‌کنیم (محبوب‌ترین conversion).

In [ ]:
import os, time
from pathlib import Path

# Cache on Drive for fast re-runs
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    CACHE = Path('/content/drive/MyDrive/argo_models')
    print('💾 Drive cache:', CACHE)
except Exception:
    CACHE = Path('/content/models')
    print('💾 Local cache:', CACHE)

CACHE.mkdir(parents=True, exist_ok=True)

REPO_ID = 'bartowski/huihui-ai_Qwen3-14B-abliterated-GGUF'
MODEL_FILE = 'huihui-ai_Qwen3-14B-abliterated-Q4_K_M.gguf'
LOCAL = CACHE / MODEL_FILE
EXPECTED_SIZE = 9_000_000_000

if LOCAL.exists() and LOCAL.stat().st_size > EXPECTED_SIZE * 0.95:
    print(f'✅ Model already cached: {LOCAL} ({LOCAL.stat().st_size/1e9:.2f} GB)')
else:
    print(f'⏬ Downloading {MODEL_FILE} from {REPO_ID} (~9GB, first run only)...')
    t0 = time.time()
    from huggingface_hub import hf_hub_download
    try:
        hf_hub_download(
            repo_id=REPO_ID,
            filename=MODEL_FILE,
            local_dir=str(CACHE),
        )
        print(f'✅ Downloaded in {time.time()-t0:.0f}s')
    except Exception as e:
        print(f'❌ {REPO_ID} failed: {e}')
        print('Trying fallback: mlabonne version via bartowski...')
        REPO_ID = 'bartowski/mlabonne_Qwen3-14B-abliterated-GGUF'
        MODEL_FILE = 'mlabonne_Qwen3-14B-abliterated-Q4_K_M.gguf'
        LOCAL = CACHE / MODEL_FILE
        hf_hub_download(
            repo_id=REPO_ID,
            filename=MODEL_FILE,
            local_dir=str(CACHE),
        )
        print(f'✅ Downloaded fallback in {time.time()-t0:.0f}s')

print(f'📁 Model: {LOCAL}')
print(f'📊 Size:  {LOCAL.stat().st_size/1e9:.2f} GB')
MODEL_PATH = str(LOCAL)

## 4) دانلود Argo (از GitHub)

In [ ]:
import os, subprocess

os.chdir('/content')
ARGO_REPO = 'https://github.com/minam67889-bit/Argo.git'

if not os.path.exists('/content/argo/app/main.py'):
    print(f'⏬ Cloning Argo from {ARGO_REPO}...')
    subprocess.run(['git', 'clone', '--depth=1', ARGO_REPO, 'argo'], check=True)
else:
    print('🔄 Updating Argo...')
    os.chdir('/content/argo')
    subprocess.run(['git', 'pull'], check=False)
    os.chdir('/content')

os.chdir('/content/argo')
print('✅ Argo ready at /content/argo')

## 5) راه‌اندازی llama-server (سرور مدل)

از `llama_server.py` استفاده می‌کنیم — یه wrapper OpenAI-compatible روی `llama-cpp-python`.

**نکته مهم:** context روی **8K** گذاشتیم تا با T4 جا بشه. می‌تونی بعداً تغییرش بدی (اما OOM میشه).

In [ ]:
import os, subprocess, time, requests

LLAMA_PORT = 8080
CONTEXT_SIZE = 8192  # 8K — safe for T4 with 14B Q4_K_M

# Clean up any previous instance (be thorough)
print('🧹 Cleaning up any previous server...')
subprocess.run(['pkill', '-9', '-f', 'llama_server.py'], check=False)
subprocess.run(['pkill', '-9', '-f', 'argo/colab/llama'], check=False)
time.sleep(2)
# Try to free the port too
try:
    subprocess.run(['fuser', '-k', f'{LLAMA_PORT}/tcp'], capture_output=True, check=False, timeout=5)
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass  # fuser not available or hung
time.sleep(1)
# Remove old port file
try:
    os.remove('/content/llama.port')
except FileNotFoundError:
    pass

env = os.environ.copy()
env['MODEL_PATH'] = MODEL_PATH
env['N_CTX'] = str(CONTEXT_SIZE)
env['N_GPU_LAYERS'] = '-1'
env['CHAT_FORMAT'] = 'chatml'
env['LLAMA_PORT'] = str(LLAMA_PORT)
env['LLAMA_HOST'] = '127.0.0.1'
env['MODEL_NAME'] = 'qwen3-14b-abliterated'

print(f'🚀 Starting llama-server...')
print(f'   Model: {MODEL_PATH}')
print(f'   Context: {CONTEXT_SIZE} tokens (8K)')
print(f'   GPU: all layers')
print()

log = open('/content/llama.log', 'w')
proc = subprocess.Popen(
    [sys.executable, '/content/argo/colab/llama_server.py'],
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f'   PID: {proc.pid}')
print('⏳ Loading model into VRAM (20-60s)...')

# Wait for the actual port file — server writes it once port is bound
# Model loading can take 1-3 min on T4, so we wait up to 4 min
actual_port = None
for i in range(240):
    if os.path.exists('/content/llama.port'):
        try:
            actual_port = int(open('/content/llama.port').read().strip())
            break
        except Exception:
            pass
    if proc.poll() is not None:
        print(f'\n❌ Server process died with code {proc.returncode}')
        print('Log:')
        print(open('/content/llama.log').read())
        raise SystemExit(1)
    time.sleep(1)

if actual_port is None:
    print('\n❌ Server did not bind any port within 4 minutes')
    print('Log:')
    print(open('/content/llama.log').read())
    raise SystemExit(1)

if actual_port != LLAMA_PORT:
    print(f'   (port {LLAMA_PORT} was busy, server chose {actual_port} instead)')

# Wait for health
print(f'⏳ Server bound to port {actual_port}, waiting for /health...')
for i in range(60):
    try:
        r = requests.get(f'http://127.0.0.1:{actual_port}/health', timeout=2)
        if r.status_code == 200:
            print(f'\n✅ llama-server ready after {i}s on port {actual_port}!')
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('\n❌ /health did not respond')
    print('Log:')
    print(open('/content/llama.log').read())
    raise SystemExit(1)

LLAMA_PORT = actual_port

# Test
print('\n🧪 Quick test...')
r = requests.post(
    f'http://127.0.0.1:{LLAMA_PORT}/v1/chat/completions',
    json={
        'model': 'qwen3-14b-abliterated',
        'messages': [{'role': 'user', 'content': 'در یک جمله بگو امروز چه روزیه؟'}],
        'max_tokens': 100,
    },
    timeout=120,
)
r.raise_for_status()
data = r.json()
print(f'✅ Response: {data["choices"][0]["message"]["content"]!r}')

## 6) راه‌اندازی Argo UI

In [ ]:
import os, subprocess, time, requests

ARGO_PORT = 8000
# LLAMA_PORT is from previous cell

subprocess.run(['pkill', '-9', '-f', 'argo/app.main'], check=False)
time.sleep(2)

env = os.environ.copy()
env['LLM_API_KEY'] = 'sk-no-key-required'
env['LLM_BASE_URL'] = f'http://127.0.0.1:{LLAMA_PORT}/v1'
env['LLM_MODEL'] = 'qwen3-14b-abliterated'
env['AGENT_TEMPERATURE'] = '0.2'
env['AGENT_MAX_TOKENS'] = '2048'  # reduced to save VRAM
env['AGENT_MAX_STEPS'] = '30'
env['ARGO_PORT'] = str(ARGO_PORT)
env['ARGO_HOST'] = '127.0.0.1'
env['ARGO_WORKSPACE'] = '/content/workspace'
os.makedirs('/content/workspace', exist_ok=True)

os.chdir('/content/argo')
print(f'🚀 Starting Argo on port {ARGO_PORT}, pointing to llama-server:{LLAMA_PORT}')
argo_log = open('/content/argo.log', 'w')
argo_proc = subprocess.Popen(
    [sys.executable, '-m', 'app.main'],
    env=env,
    stdout=argo_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f'   PID: {argo_proc.pid}')
print('⏳ Waiting for Argo...')

for i in range(30):
    try:
        r = requests.get(f'http://127.0.0.1:{ARGO_PORT}/api/health', timeout=2)
        if r.status_code == 200:
            print(f'\n✅ Argo ready after {i}s')
            h = r.json()
            print(f'   Model:  {h["model"]}')
            print(f'   API:    {h["base_url"]}')
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('❌ Argo failed. Log:')
    print(open('/content/argo.log').read())
    raise SystemExit(1)

## 7) Cloudflare Tunnel — URL عمومی

In [ ]:
import os, subprocess, time, re

TUNNEL_PORT = 8000

if not os.path.exists('/usr/local/bin/cloudflared'):
    print('⏬ Downloading cloudflared...')
    subprocess.run([
        'wget', '-q', '-O', '/tmp/cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    ], check=True)
    subprocess.run(['chmod', '+x', '/tmp/cloudflared'], check=True)
    os.makedirs('/usr/local/bin', exist_ok=True)
    subprocess.run(['mv', '/tmp/cloudflared', '/usr/local/bin/cloudflared'], check=True)
    print('✅ cloudflared installed')

subprocess.run(['pkill', '-9', '-f', 'cloudflared'], check=False)
time.sleep(1)

print('🌐 Starting tunnel...')
tunnel_log = open('/content/tunnel.log', 'w')
tunnel_proc = subprocess.Popen([
    'cloudflared', 'tunnel', '--no-autoupdate',
    '--url', f'http://127.0.0.1:{TUNNEL_PORT}',
    '--metrics', '127.0.0.1:0',
], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)

print('⏳ Waiting for URL...')
url = None
for i in range(60):
    time.sleep(1)
    try:
        text = tunnel_log.read() if not tunnel_log.closed else open('/content/tunnel.log').read()
    except Exception:
        text = open('/content/tunnel.log').read()
    m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', text)
    if m:
        url = m.group(1)
        break

if not url:
    print('❌ Tunnel failed. Log:')
    print(open('/content/tunnel.log').read())
    raise SystemExit(1)

print()
print('=' * 70)
print('✅ Argo is LIVE!')
print('=' * 70)
print()
print(f'🌐 URL: {url}')
print()
print('OpenAI-compatible clients:')
print(f'   Base URL: {url}/v1')
print(f'   API Key:  sk-no-key-required')
print(f'   Model:    qwen3-14b-abliterated')
print()
print('=' * 70)
print('To stop: Runtime → Interrupt')
print('=' * 70)

## 🎉 تموم!

روی URL بالا بزن و:

- **حالت چت**: گفتگوی آزاد با Qwen3-14B-abliterated (uncensored)
- **حالت ایجنت**: تسک بده، ایجنت با bash + فایل کار می‌کنه

### عیب‌یابی

اگه runtime ری‌استارت شد، log رو ببین:

```bash
!tail -30 /content/llama.log   # مدل — مهم‌ترین
!tail -30 /content/argo.log    # Argo
```

اگه خطای OOM دیدی، `CONTEXT_SIZE` رو توی سلول 5 کمتر کن (مثلاً 4096).